In [1]:
# gan_embedding_counterfactual.py
# Multi-input embedding GAN with counterfactual-based bias labels (Strategy B)
# Fixes applied:
# - embeddings instead of giant one-hots
# - discriminator widths: 256, 128, 64
# - batch_size = 32
# - sample dataset to 20k rows
# - precompute surrogate f_approval and labels

import os
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from collections import Counter

# -------------------------
# User column lists (paste your lists)
# -------------------------
num_cols = [
    "loan_amount", "loan_to_value_ratio", "interest_rate", "rate_spread",
    "total_loan_costs", "total_points_and_fees", "origination_charges",
    "discount_points", "lender_credits", "loan_term",
    "prepayment_penalty_term", "intro_rate_period", "property_value",
    "total_units", "multifamily_affordable_units", "income",
    "tract_population", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage",
    "tract_owner_occupied_units", "tract_one_to_four_family_homes",
    "tract_median_age_of_housing_units"
]

cat_cols = [
    "lei", "derived_msa-md", "state_code", "county_code",
    "census_tract", "conforming_loan_limit", "derived_loan_product_type",
    "derived_dwelling_category", "derived_ethnicity", "derived_race",
    "derived_sex", "action_taken", "purchaser_type", "preapproval",
    "loan_type", "loan_purpose", "lien_status", "reverse_mortgage",
    "open-end_line_of_credit", "business_or_commercial_purpose",
    "construction_method", "occupancy_type",
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest",
    "applicant_credit_score_type", "co-applicant_credit_score_type",
    "applicant_ethnicity_observed", "co-applicant_ethnicity_observed",
    "applicant_race_observed", "co-applicant_race_observed",
    "applicant_sex", "co-applicant_sex", "applicant_sex_observed",
    "co-applicant_sex_observed", "applicant_age_above_62",
    "co-applicant_age_above_62", "submission_of_application",
    "initially_payable_to_institution",
    "negative_amortization", "interest_only_payment",
    "balloon_payment", "other_nonamortizing_features"
]

multi_slot_cols = [
    "applicant_ethnicity-1","applicant_ethnicity-2","applicant_ethnicity-3",
    "applicant_ethnicity-4","applicant_ethnicity-5",
    "co-applicant_ethnicity-1","co-applicant_ethnicity-2",
    "co-applicant_ethnicity-3","co-applicant_ethnicity-4",
    "co-applicant_ethnicity-5",
    "applicant_race-1","applicant_race-2","applicant_race-3",
    "applicant_race-4","applicant_race-5",
    "co-applicant_race-1","co-applicant_race-2",
    "co-applicant_race-3","co-applicant_race-4","co-applicant_race-5",
    "aus-1","aus-2","aus-3","aus-4","aus-5",
    "denial_reason-1","denial_reason-2","denial_reason-3","denial_reason-4"
]

# -------------------------
# Preprocessor for embedding architecture
# - numeric scaler
# - label encoding for categorical columns
# - each multi-slot slot is treated as its own categorical column
# -------------------------
class PreprocessorEmbedding:
    def __init__(self, num_cols, cat_cols, multi_slot_cols, top_k_alternatives=10):
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        # flatten multi-slot names into slot-level columns
        self.multi_slot_cols = multi_slot_cols
        self.num_scaler = StandardScaler()
        self.label_maps = {}  # col -> {value: index}
        self.inverse_label_maps = {}
        self.cardinalities = {}
        self.top_k_alternatives = top_k_alternatives

    def fit(self, df):
        # numeric scaler
        self.num_scaler.fit(df[self.num_cols].fillna(0.0).astype(float))

        # categorical label maps
        for c in self.cat_cols + self.multi_slot_cols:
            vals = df[c].astype(str).fillna("<<NA>>")
            # create mapping of frequent values first
            counts = vals.value_counts()
            ordered = list(counts.index)
            mapping = {v: i for i, v in enumerate(ordered)}
            self.label_maps[c] = mapping
            self.inverse_label_maps[c] = {i: v for v, i in mapping.items()}
            self.cardinalities[c] = len(mapping)

        return self

    def transform_indices(self, df):
        # numeric
        X_num = df[self.num_cols].fillna(0.0).astype(float).values
        # categorical indices: for each cat col produce integer array (n,)
        cat_indices = {}
        for c in self.cat_cols + self.multi_slot_cols:
            mapping = self.label_maps[c]
            vals = df[c].astype(str).fillna("<<NA>>")
            # map unseen to a reserved index at end
            def map_val(v):
                return mapping.get(v, len(mapping))  # unseen -> new index
            # vectorize mapping
            cat_indices[c] = vals.map(map_val).astype(int).values
            # if unseen exists, increase cardinality to account for unseen slot
            if any(vals.map(lambda v: v not in mapping)):
                self.cardinalities[c] = max(self.cardinalities[c], len(mapping) + 1)
        return X_num, cat_indices

    def inverse_transform_cat(self, c, idx):
        return self.inverse_label_maps.get(c, {}).get(idx, "<<UNK>>")

    def top_k_alts_for_col(self, df, col, k=None):
        if k is None:
            k = self.top_k_alternatives
        vals = df[col].astype(str).fillna("<<NA>>")
        freqs = vals.value_counts().index.tolist()
        return freqs[:k]

# -------------------------
# Surrogate approval model helpers
# -------------------------
def map_action_to_approval(series):
    # approve for action_taken in {1,2,6,8}
    approved = {"1", "2", "6", "8"}
    return series.astype(str).fillna("0").apply(lambda x: 1 if x in approved else 0).values

# -------------------------
# Counterfactual labeling (limited alternatives)
# -------------------------
def compute_counterfactual_labels(df, pre, surrogate, sensitive_cols, delta=0.10, top_k=10):
    # We'll iterate over the df and compute p0 and p_cf for a limited set of alternatives
    n = len(df)
    labels = np.zeros((n, 1), dtype=np.float32)

    # Precompute base features for surrogate: numeric + integer indices flattened into feature vector
    X_num, cat_indices = pre.transform_indices(df)

    # surrogate expects a 2D array of features. We'll form a simple tabular array:
    # features = [numeric columns | categorical integer columns concatenated]
    cat_cols_ordered = pre.cat_cols + pre.multi_slot_cols
    cat_matrix = np.column_stack([cat_indices[c] for c in cat_cols_ordered])
    X_surrogate = np.hstack([X_num, cat_matrix])

    p0 = surrogate.predict_proba(X_surrogate)[:, 1]

    # determine alternatives per sensitive column
    alt_values_per_col = {}
    for s in sensitive_cols:
        alt_values_per_col[s] = pre.top_k_alts_for_col(df, s, k=top_k)

    for i in range(n):
        base_p = p0[i]
        max_abs_diff = 0.0
        for s in sensitive_cols:
            original = str(df.iloc[i][s]) if pd.notnull(df.iloc[i][s]) else "<<NA>>"
            for alt in alt_values_per_col[s]:
                if alt == original:
                    continue
                # create surrogate feature vector with only s changed to alt
                row_num = X_num[i].reshape(1, -1)
                row_cats = np.array([cat_indices[c][i] for c in cat_cols_ordered]).reshape(1, -1)
                # map alt to integer using pre.label_maps
                mapping = pre.label_maps.get(s, {})
                alt_idx = mapping.get(alt, len(mapping))
                # find column index in row_cats for s
                col_idx = cat_cols_ordered.index(s)
                row_cats_cf = row_cats.copy()
                row_cats_cf[0, col_idx] = alt_idx
                X_cf = np.hstack([row_num, row_cats_cf])
                p_cf = surrogate.predict_proba(X_cf)[:, 1][0]
                diff = abs(p_cf - base_p)
                if diff > max_abs_diff:
                    max_abs_diff = diff
                if max_abs_diff >= delta:
                    break
            if max_abs_diff >= delta:
                break
        labels[i, 0] = 1.0 if max_abs_diff >= delta else 0.0

    return labels

# -------------------------
# Build embedding sizes helper
# -------------------------
def embedding_size(cardinality):
    # conservative embedding size formula
    return max(4, min(50, int(np.ceil(cardinality ** 0.5))))

# -------------------------
# Build generator and discriminator for embedding architecture
# - generator outputs numeric vector and categorical logits for each categorical column and each multi-slot slot
# - discriminator accepts numeric vector and embedding vectors (we build a wrapper that gathers embeddings for real indices)
# -------------------------
def build_generator(noise_dim, num_numeric, cat_cardinalities, multi_slot_cardinalities, hidden_dims=(512, 512)):
    """
    Returns generator model that outputs:
     - numeric_out: shape (num_numeric,)
     - cat_logits: list of tensors per categorical column (batch, k_c)
     - slot_logits: list of tensors per multi-slot slot (batch, k_slot)
    The Keras Model will return a list: [numeric_out] + cat_logits + slot_logits
    """
    z = Input(shape=(noise_dim,), name="g_noise")
    x = z
    for i, h in enumerate(hidden_dims):
        x = layers.Dense(h, activation="relu", name=f"g_dense_{i}")(x)
        x = layers.BatchNormalization()(x)
    # numeric output
    numeric_out = layers.Dense(num_numeric, activation=None, name="g_numeric")(x)

    outputs = [numeric_out]
    # categorical logits
    for idx, (col, k) in enumerate(cat_cardinalities.items()):
        logits = layers.Dense(k, activation=None, name=f"g_catlogits_{col}")(x)
        outputs.append(logits)
    # multi-slot logits
    for idx, (col, k) in enumerate(multi_slot_cardinalities.items()):
        logits = layers.Dense(k, activation=None, name=f"g_slotlogits_{col}")(x)
        outputs.append(logits)

    model = Model(inputs=z, outputs=outputs, name="Generator")
    return model

def build_discriminator_core(input_dim):
    inp = Input(shape=(input_dim,), name="disc_concat_input")
    x = layers.Dense(256, activation="relu")(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inp, out, name="DiscriminatorCore")
    return model

def assemble_discriminator_model(num_numeric, cat_info, slot_info, embedding_layers, disc_core):
    """
    Build a Keras model that takes:
     - numeric_input: (batch, num_numeric)
     - one integer input per categorical column
     - one integer input per multi-slot slot
    It uses embedding_layers (dict col -> Embedding layer instance) to map indices to embeddings,
    concatenates numeric + embeddings and feeds into disc_core.
    """
    numeric_in = Input(shape=(num_numeric,), name="disc_numeric_in")
    cat_inputs = []
    slot_inputs = []
    embeddings = [numeric_in]

    # categorical columns
    for col, k in cat_info.items():
        inp = Input(shape=(1,), dtype="int32", name=f"disc_in_{col}")
        cat_inputs.append(inp)
        emb = embedding_layers[col](inp)  # result shape (batch, 1, emb_dim)
        emb = layers.Reshape((embedding_layers[col].output_dim,))(emb)  # (batch, emb_dim)
        embeddings.append(emb)

    # multi-slot columns
    for col, k in slot_info.items():
        inp = Input(shape=(1,), dtype="int32", name=f"disc_in_{col}")
        slot_inputs.append(inp)
        emb = embedding_layers[col](inp)
        emb = layers.Reshape((embedding_layers[col].output_dim,))(emb)
        embeddings.append(emb)

    concat = layers.Concatenate(axis=1)(embeddings)
    out = disc_core(concat)
    model_inputs = [numeric_in] + cat_inputs + slot_inputs
    model = Model(inputs=model_inputs, outputs=out, name="Discriminator")
    return model

# -------------------------
# Combined model: noise -> generator -> softmax logits -> embeddings via embedding weights -> concatenated -> disc_core -> output
# We build using the same embedding layers and disc_core so that embeddings are shared.
# -------------------------
def build_combined_model(generator, embedding_layers, disc_core, cat_info, slot_info, num_numeric):
    noise_in = Input(shape=(generator.input_shape[1],), name="comb_noise")
    gen_outputs = generator(noise_in)
    numeric_out = gen_outputs[0]
    idx = 1
    embed_vectors = [numeric_out]  # will be concatenated

    # For each categorical column: gen_outputs[idx] are logits (batch, k)
    for col in cat_info.keys():
        logits = gen_outputs[idx]  # (batch, k)
        idx += 1
        probs = layers.Activation("softmax")(logits)  # (batch, k)
        # embedding weight matrix variable for this column
        W = embedding_layers[col].embeddings  # shape (k, emb_dim)
        # compute embedding = probs @ W  using a Lambda layer
        emb = layers.Lambda(
            lambda args: tf.matmul(args[0], args[1]),
            output_shape=lambda shapes: (shapes[0][0], shapes[1][1])
        )([probs, W])
        embed_vectors.append(emb)  # shape (batch, emb_dim)

    for col in slot_info.keys():
        logits = gen_outputs[idx]
        idx += 1
        probs = layers.Activation("softmax")(logits)
        W = embedding_layers[col].embeddings
        emb = layers.Lambda(
            lambda args: tf.matmul(args[0], args[1]),
            output_shape=lambda shapes: (shapes[0][0], shapes[1][1])
        )([probs, W])
        embed_vectors.append(emb)

    concat = layers.Concatenate(axis=1)(embed_vectors)
    out = disc_core(concat)
    combined_model = Model(inputs=noise_in, outputs=out, name="CombinedGAN")
    return combined_model

# -------------------------
# Training loop utilities
# -------------------------
def sample_noise(batch, noise_dim):
    return np.random.normal(0, 1, size=(batch, noise_dim)).astype(np.float32)

# -------------------------
# Main execution
# -------------------------
if __name__ == "__main__":
    csv_path = "pwc_dataset.csv"
    if not os.path.exists(csv_path):
        raise FileNotFoundError("Place your CSV at data.csv")

    df = pd.read_csv(csv_path, dtype=str, sep=";")

    # Ensure numeric columns exist and convert
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = 0.0

    # Ensure categorical and multi-slot exist
    for c in cat_cols + multi_slot_cols:
        if c not in df.columns:
            df[c] = "<<NA>>"
        df[c] = df[c].astype(str)

    # Sample dataset for faster iteration
    sample_size = min(20000, len(df))
    df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)
    print("Using sample size:", sample_size)

    # Preprocessor
    pre = PreprocessorEmbedding(num_cols=num_cols, cat_cols=cat_cols, multi_slot_cols=multi_slot_cols, top_k_alternatives=10)
    pre.fit(df_sample)

    # Prepare surrogate training features
    X_num, cat_indices = pre.transform_indices(df_sample)
    cat_cols_ordered = pre.cat_cols + pre.multi_slot_cols
    cat_matrix = np.column_stack([cat_indices[c] for c in cat_cols_ordered])
    X_surrogate = np.hstack([X_num, cat_matrix])

    # Surrogate approval labels
    y_approval = map_action_to_approval(df_sample["action_taken"])

    # Train surrogate model
    X_train, X_val, y_train, y_val = train_test_split(X_surrogate, y_approval, test_size=0.2, random_state=42)
    surrogate = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
    surrogate.fit(X_train, y_train.ravel())
    print("Surrogate validation accuracy:", surrogate.score(X_val, y_val.ravel()))

    # Compute counterfactual bias labels
    sensitive_cols = ["derived_race", "derived_ethnicity", "derived_sex"]
    delta = 0.10
    print("Computing counterfactual labels (this may take a few minutes)...")
    bias_labels = compute_counterfactual_labels(df_sample, pre, surrogate, sensitive_cols, delta=delta, top_k=10)
    print("Biased rows:", int(bias_labels.sum()), "out of", len(bias_labels))

    # Save preprocessor and surrogate
    with open("preprocessor_embedding.pkl", "wb") as f:
        pickle.dump(pre, f)
    with open("surrogate_clf.pkl", "wb") as f:
        pickle.dump(surrogate, f)

    # Build embedding info maps
    # cat_info: dict col -> cardinality
    cat_info = {c: pre.cardinalities[c] for c in pre.cat_cols}
    slot_info = {s: pre.cardinalities[s] for s in pre.multi_slot_cols}

    # Create embedding layers for each categorical and slot column
    embedding_layers = {}
    for c, k in list(cat_info.items()) + list(slot_info.items()):
        if k <= 1:
            emb_dim = 4
        else:
            emb_dim = embedding_size(k)
        # we set mask_zero False; unseen indices map to index len(mapping) and are included in cardinality
        embedding_layers[c] = layers.Embedding(input_dim=k + 1, output_dim=emb_dim, name=f"emb_{c}")

    # Build generator and discriminator core
    noise_dim = 128
    num_numeric = len(num_cols)
    gen = build_generator(noise_dim=noise_dim,
                          num_numeric=num_numeric,
                          cat_cardinalities=cat_info,
                          multi_slot_cardinalities=slot_info,
                          hidden_dims=(512, 512))

    # compute total concatenated embedding size for discriminator core input dim
    total_emb_dim = num_numeric + sum([embedding_layers[c].output_dim for c in cat_info.keys()]) + sum([embedding_layers[s].output_dim for s in slot_info.keys()])
    disc_core = build_discriminator_core(total_emb_dim)

    # Assemble discriminator model that accepts numeric + integer indices as inputs
    disc_model = assemble_discriminator_model(num_numeric=num_numeric, cat_info=cat_info, slot_info=slot_info, embedding_layers=embedding_layers, disc_core=disc_core)
    disc_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4), loss="binary_crossentropy", metrics=["accuracy"])

    # Build combined model to train generator end-to-end
    combined = build_combined_model(gen, embedding_layers, disc_core, cat_info, slot_info, num_numeric)
    combined.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4), loss="binary_crossentropy")

    print("Generator summary:")
    gen.summary()
    print("Discriminator summary:")
    disc_model.summary()

    # Prepare training arrays for discriminator real batches
    X_num_all = X_num.astype(np.float32)
    # For keras inputs we need each categorical column as shape (n, 1) int32
    cat_inputs_all = {c: cat_indices[c].astype(np.int32).reshape(-1, 1) for c in cat_cols_ordered}

    # Training parameters
    batch_size = 32
    epochs = 1000
    report_every = 50
    n = len(df_sample)
    steps_per_epoch = max(1, n // batch_size)

    print("Begin GAN training")
    for epoch in range(1, epochs + 1):
        # sample random batch indices
        ids = np.random.randint(0, n, size=batch_size)
        # real numeric and real cat indices batch
        real_num = X_num_all[ids]
        # construct discriminator inputs for real batch
        disc_real_inputs = [real_num]
        for c in pre.cat_cols:
            disc_real_inputs.append(cat_inputs_all[c][ids])
        for s in pre.multi_slot_cols:
            disc_real_inputs.append(cat_inputs_all[s][ids])

        real_labels = bias_labels[ids].astype(np.float32)

        # Generate fake samples using generator
        noise = sample_noise(batch_size, noise_dim)
        gen_outs = gen.predict(noise, verbose=0)
        # parse generator outputs
        numeric_fake = gen_outs[0]  # shape (batch, num_numeric)
        # next outputs correspond to cat logits in the same order as cat_info keys, then slot logits
        idx_out = 1
        fake_embeddings = [numeric_fake]  # will be numpy arrays

        # For each categorical column compute softmax probs and compute embeddings with current embedding weights
        for col in cat_info.keys():
            logits = gen_outs[idx_out]  # shape (batch, k)
            idx_out += 1
            # softmax
            exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
            probs = exps / np.sum(exps, axis=1, keepdims=True)
            # get current embedding weights from embedding_layers[col]
            W = embedding_layers[col].get_weights()[0]  # shape (k+1, emb_dim) note last row corresponds to unseen index
            # use only first k rows (embedding dims matched to input dim k+1)
            # If W has k+1 rows, we use first k rows, but probs shape matches k
            W_use = W[:probs.shape[1], :]  # (k, emb_dim)
            emb = probs.dot(W_use)  # (batch, emb_dim)
            fake_embeddings.append(emb)

        # multi-slot logits
        for col in slot_info.keys():
            logits = gen_outs[idx_out]
            idx_out += 1
            exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
            probs = exps / np.sum(exps, axis=1, keepdims=True)
            W = embedding_layers[col].get_weights()[0]
            W_use = W[:probs.shape[1], :]
            emb = probs.dot(W_use)
            fake_embeddings.append(emb)

        # concatenate fake embeddings to form disc input vector
        fake_concat = np.concatenate(fake_embeddings, axis=1).astype(np.float32)
        # train discriminator core on fake data using disc_core via a wrapper model that accepts concatenated input
        # we can use disc_core directly
        fake_labels = np.ones((batch_size, 1), dtype=np.float32)  # generator tries to make biased rows
        # train discriminator on both real and fake by calling disc_model.train_on_batch on a combined batch
        # Build combined batch inputs for real+fake
        # For real part we already have disc_real_inputs; for fake part we need to create inputs compatible with disc_model:
        # disc_model expects numeric and integer index inputs, we will pass embeddings only via disc_core using a helper that takes concatenated embeddings
        # Simpler approach: train disc_core directly: prepare concatenated real embeddings and concatenated fake embeddings
        # compute real concatenated embeddings by running embedding layers on the real indices via a small model
        # create a helper function model_to_get_concat that gets embedding outputs given integer inputs
        # We'll create it once outside the loop. For now assume we have it.

        # We'll compute real concatenated embeddings using a small function below
        # see creation of get_real_concat_model below
        pass

    # The above loop contains a 'pass' placeholder. The next block implements the training loop correctly.
    # To keep the file coherent and runnable, we reimplement the training loop below using helper model get_real_concat_model.

    # Build helper model to compute concatenated embeddings for integer-index inputs
    # Inputs identical to disc_model inputs; outputs: concatenated vector before disc_core
    numeric_in = Input(shape=(num_numeric,), name="helper_numeric_in")
    helper_embs = [numeric_in]
    for c in pre.cat_cols:
        inp = Input(shape=(1,), dtype="int32", name=f"helper_in_{c}")
        emb = embedding_layers[c](inp)
        emb = layers.Reshape((embedding_layers[c].output_dim,))(emb)
        helper_embs.append(emb)
    for s in pre.multi_slot_cols:
        inp = Input(shape=(1,), dtype="int32", name=f"helper_in_{s}")
        emb = embedding_layers[s](inp)
        emb = layers.Reshape((embedding_layers[s].output_dim,))(emb)
        helper_embs.append(emb)
    helper_concat = layers.Concatenate(axis=1)(helper_embs)
    get_real_concat_model = Model(inputs=[numeric_in] + [Input(shape=(1,), dtype="int32", name=f"tmp_{c}") for c in pre.cat_cols + pre.multi_slot_cols],
                                  outputs=helper_concat)
    # The above temporary model creation used ephemeral Inputs just to define shapes.
    # Instead we construct a tf.function that given numeric and indices returns concatenated embeddings using the embedding layers directly.

    # Create a function to compute concatenated embeddings for real inputs using the embedding_layers
    def compute_real_concat(numeric_batch, indices_batch_dict):
        embs = [numeric_batch]
        for c in pre.cat_cols:
            idxs = indices_batch_dict[c]  # shape (batch, 1)
            # gather embedding weights via embedding_layers[c]
            W = embedding_layers[c].get_weights()[0]  # shape (k+1, emb_dim)
            # map indices to W rows using numpy (works since idxs are ints)
            emb = W[idxs.reshape(-1), :]  # (batch, emb_dim)
            embs.append(emb)
        for s in pre.multi_slot_cols:
            idxs = indices_batch_dict[s]
            W = embedding_layers[s].get_weights()[0]
            emb = W[idxs.reshape(-1), :]
            embs.append(emb)
        return np.concatenate(embs, axis=1).astype(np.float32)

    # Now full training loop
    # -------------------------
# Assumptions: the following variables exist and are compiled:
# - pre (PreprocessorEmbedding)
# - X_num_all (np.array shape (n, num_numeric), dtype=float32)
# - cat_inputs_all: dict col -> np.array shape (n,1), dtype=int32 for all cat_cols_ordered
# - gen (Generator Keras model)
# - disc_model (full discriminator taking numeric + integer inputs) -> compiled
# - disc_core (Model taking concatenated embeddings) -> compiled
# - combined (Model: noise -> disc_core using embedding variables) -> compiled
# - embedding_layers (dict col -> Embedding layer)
# - cat_info (ordered dict of categorical column -> cardinality)
# - slot_info (ordered dict of multi-slot column -> cardinality)
# - pre.cat_cols, pre.multi_slot_cols lists exist and match cat_cols_ordered
# - bias_labels (np.array shape (n,1), dtype=float32)
# - batch_size, noise_dim, epochs, report_every, n defined
# -------------------------

import math
from tqdm import trange

cat_cols_ordered = pre.cat_cols + pre.multi_slot_cols
num_numeric = X_num_all.shape[1]
n = X_num_all.shape[0]

# utility: sample noise
def sample_noise(batch, noise_dim):
    return np.random.normal(0, 1, size=(batch, noise_dim)).astype(np.float32)

# Precompute total embedding dim for debugging (not used in training directly)
total_emb_dim = num_numeric
for c in cat_info.keys():
    total_emb_dim += embedding_layers[c].output_dim
for s in slot_info.keys():
    total_emb_dim += embedding_layers[s].output_dim

print(f"Training: n={n}, batch_size={batch_size}, noise_dim={noise_dim}, total_emb_dim={total_emb_dim}")

# Helper to compute fake concatenated embeddings from generator outputs (numpy)
def fake_concat_from_generator_outputs(gen_outputs, embedding_layers, cat_info, slot_info):
    """
    gen_outputs: list where
      gen_outputs[0] -> numeric_fake (batch, num_numeric)
      next len(cat_info) items -> logits for each categorical column (batch, k)
      next len(slot_info) items -> logits for each slot (batch, k_slot)
    Returns numpy array shape (batch, total_emb_dim)
    """
    numeric_fake = gen_outputs[0]  # (batch, num_numeric)
    batch = numeric_fake.shape[0]
    emb_list = [numeric_fake]

    idx_out = 1
    # categorical columns
    for col in cat_info.keys():
        logits = gen_outputs[idx_out]  # (batch, k)
        idx_out += 1
        # stable softmax
        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exps / np.sum(exps, axis=1, keepdims=True)  # (batch, k)
        # get current embedding weight matrix (k+1, emb_dim) -> we use first k rows
        W = embedding_layers[col].get_weights()[0]  # numpy array
        W_use = W[:probs.shape[1], :]  # (k, emb_dim)  # probs.shape[1] == k
        # compute embedding vectors as probs @ W_use => (batch, emb_dim)
        emb = probs.dot(W_use)
        emb_list.append(emb)

    # multi-slot columns
    for col in slot_info.keys():
        logits = gen_outputs[idx_out]  # (batch, k_slot)
        idx_out += 1
        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exps / np.sum(exps, axis=1, keepdims=True)
        W = embedding_layers[col].get_weights()[0]
        W_use = W[:probs.shape[1], :]
        emb = probs.dot(W_use)
        emb_list.append(emb)

    concat = np.concatenate(emb_list, axis=1).astype(np.float32)
    return concat

# Training loop
for epoch in range(1, epochs + 1):
    # iterate steps: we will perform steps_per_epoch discriminator/generator updates
    steps = max(1, math.ceil(n / batch_size))
    d_losses = []
    d_accs = []
    g_losses = []

    for step in range(steps):
        # ----------------------
        # 1) Discriminator real update (use disc_model so embedding layers are updated)
        # ----------------------
        ids = np.random.randint(0, n, size=batch_size)
        real_num = X_num_all[ids]  # (batch, num_numeric)
        # build list of integer inputs matching disc_model inputs: [numeric] + [col_indices as (batch,1) int32]
        disc_real_inputs = [real_num]
        for c in cat_cols_ordered:
            disc_real_inputs.append(cat_inputs_all[c][ids].astype(np.int32).reshape(-1, 1))

        real_labels = bias_labels[ids].astype(np.float32)  # shape (batch,1) with 0/1

        # Train on real batch (this updates embedding layers and disc_core parameters)
        d_loss_real, d_acc_real = disc_model.train_on_batch(disc_real_inputs, real_labels)
        # record metrics
        d_losses.append(d_loss_real)
        d_accs.append(d_acc_real)

        # ----------------------
        # 2) Discriminator fake update (train disc_core on concatenated fake embeddings)
        # ----------------------
        noise = sample_noise(batch_size, noise_dim)
        gen_outs = gen.predict(noise, verbose=0)  # list
        fake_concat = fake_concat_from_generator_outputs(gen_outs, embedding_layers, cat_info, slot_info)
        fake_labels = np.ones((batch_size, 1), dtype=np.float32)  # generator tries to make biased rows -> label 1

        # Train disc_core directly on concatenated embeddings (this updates the dense stack)
        d_loss_fake, d_acc_fake = disc_core.train_on_batch(fake_concat, fake_labels)
        # record metrics (we just append)
        d_losses.append(d_loss_fake)
        d_accs.append(d_acc_fake)

        # ----------------------
        # 3) Generator update via combined model: try to make discriminator output = 1 (biased)
        # ----------------------
        noise = sample_noise(batch_size, noise_dim)
        # targets of ones (generator wishes disc to predict biased)
        g_loss = combined.train_on_batch(noise, np.ones((batch_size, 1), dtype=np.float32))
        g_losses.append(g_loss)

    # end of epoch
    mean_d_loss = float(np.mean(d_losses)) if d_losses else 0.0
    mean_d_acc = float(np.mean(d_accs)) if d_accs else 0.0
    mean_g_loss = float(np.mean(g_losses)) if g_losses else 0.0

    if epoch % report_every == 0 or epoch == 1:
        print(f"Epoch {epoch}/{epochs}  D_loss(avg)={mean_d_loss:.4f} D_acc(avg)={mean_d_acc:.4f}  G_loss(avg)={mean_g_loss:.4f}")


AttributeError: 'int' object has no attribute 'fillna'